In [1]:
## from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import torchaudio 
import torchaudio.transforms as T

from datasets import load_dataset

import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from src.libri_las_lightning import LASModule
from corpus.single_word import SingleWordDataset as Dataset

import fuzzy 
from tqdm.notebook import tqdm
import pandas as pd
import yaml


In [2]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set


In [3]:
clean_dataset = Dataset(path, dev_split, None, 1)


Using custom data configuration default-6e977ecf05190e74
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


In [4]:
sampling_rate = 16000

In [5]:
path_to_config = '../config/libri/libri_las_pl_train.yaml'
config = yaml.load(open(path_to_config, 'r'), Loader=yaml.FullLoader)

In [6]:
# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
path_to_ckpt = '../librispeech_pl/checkpoints/epoch=82-step=1151126.ckpt'
model = LASModule.load_from_checkpoint(path_to_ckpt, config=config)
model.eval()

LASModule(
  (data_pipeline): Sequential(
    (0): ExtractMelFromWav(mode=fbank, num_mel_bins=40)
    (1): CMVN(mode=global, dim=2, eps=1e-10)
    (2): Postprocess()
  )
  (model): ASR(
    (encoder): Encoder(
      (layers): ModuleList(
        (0): VGGExtractor(
          (extractor): Sequential(
            (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): ReLU()
            (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (3): ReLU()
            (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
            (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (6): ReLU()
            (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (8): ReLU()
            (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          )
        )
        (1): RNNLayer(
          (layer): LSTM(1280, 512, batch_

We can see this is a character model based on the handful of errors made. Overall performance looks good.

## Check on Single Word Corpora

no language model is getting used, so things may be messy 

In [32]:
def run_inference(model, dataset):
    results = []

    if model.device.type != 'cuda' and torch.cuda.is_available():
        model = model.cuda()

    device = model.device.type

    for sample in tqdm(updated_dataset):
        true = sample["word"]
        inputs, samps = model._test_collate_fn([[torch.FloatTensor(sample['audio']), '_', true]])
        with torch.no_grad():
            ctc_output, encode_len, att_output, att_align, dec_state = model(inputs)
        if att_output is not None:
            hyp_seqs = att_output.argmax(dim=-1).tolist()
        else:
            hyp_seqs = ctc_output.argmax(dim=-1).tolist()
        transcript = model.tokenizer.decode(hyp_seqs[0])
        # transcribe speech

        results.append({'hyp': transcript,
                        'truth': true})

    return results

In [8]:
clean_dataset.dataset

Dataset({
    features: ['Unnamed: 0', 'word', 'wav_path'],
    num_rows: 200
})

In [9]:
# resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
#                    rolloff=0.9475937167399596, 
#                    resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
#     wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


In [10]:
updated_dataset = clean_dataset.dataset.map(load_audio)

Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c28c661dad041df8.arrow


In [11]:
updated_dataset

Dataset({
    features: ['Unnamed: 0', 'word', 'wav_path', 'audio', 'sampling_rate'],
    num_rows: 200
})

In [12]:
in_ = model._test_collate_fn([[torch.FloatTensor(updated_dataset[0]['audio']), '_', updated_dataset[0]['word']]])

In [13]:
in_

(Batch(features=tensor([[[ 0.2527, -0.5247, -0.2929,  ...,  0.0794, -0.0050, -0.4431],
          [-0.3459, -0.5919, -0.3264,  ...,  0.1686, -0.1235, -0.5431],
          [-0.7091, -0.5729, -0.4297,  ...,  0.0756, -0.5369, -0.5195],
          ...,
          [-1.3662, -0.9809, -1.2375,  ..., -0.6278, -0.8238, -0.6262],
          [-1.1081, -1.1787, -1.1664,  ..., -0.9264, -0.7730, -0.5167],
          [-1.1652, -1.0225, -0.8049,  ..., -0.7285, -0.7851, -0.7806]]]), feature_lengths=tensor([173], dtype=torch.int32), targets=tensor([[3, 1]]), target_lengths=tensor([2], dtype=torch.int32)),
 [[tensor([[3.0518e-05, 3.0518e-05, 3.0518e-05,  ..., 6.7139e-04, 6.7139e-04,
            6.7139e-04]]),
   '_',
   'THE']])

In [20]:
results = []

if model.device.type != 'cuda' and torch.cuda.is_available():
    model = model.cuda()

device = model.device.type

for sample in tqdm(updated_dataset):
    true = sample["word"]
    inputs, samps = model._test_collate_fn([[torch.FloatTensor(sample['audio']), '_', true]])
    with torch.no_grad():
        ctc_output, encode_len, att_output, att_align, dec_state = model(inputs)
    if att_output is not None:
        hyp_seqs = att_output.argmax(dim=-1).tolist()
    else:
        hyp_seqs = ctc_output.argmax(dim=-1).tolist()
    transcript = model.tokenizer.decode(hyp_seqs[0])
    # transcribe speech

    results.append({'hyp': transcript,
                    'truth': true})


  0%|          | 0/200 [00:00<?, ?it/s]

In [21]:
transcript

'<unk>'

In [22]:
len(inputs[0])

1

In [24]:
clean_results = pd.DataFrame(results)
clean_results['condition'] = 'clean'

In [25]:
clean_results.head(5)

,hyp,truth,condition
0,<unk>,THE,clean
1,AND,AND,clean
2,<unk>,TO,clean
3,ELSE,OF,clean
4,A,A,clean


In [26]:
top_match_guesses = clean_results[(clean_results['hyp'] == clean_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(clean_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

104 matched guesses; 52.0% correct


,hyp,truth,condition
1,AND,AND,clean
4,A,A,clean
9,IS,IS,clean
10,IT,IT,clean
11,WAS,WAS,clean


In [27]:
dmeta = fuzzy.DMetaphone()

clean_results['hyp_dmeta'] = clean_results['hyp'].apply(lambda x: dmeta(x))
clean_results['truth_dmeta'] = clean_results['truth'].apply(lambda x: dmeta(x))

In [29]:
top_match_guesses = clean_results[(clean_results['hyp_dmeta'] == clean_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(clean_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

115 matched guesses; 57.5% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
1,AND,AND,clean,"[b'ANT', None]","[b'ANT', None]"
4,A,A,clean,"[b'A', None]","[b'A', None]"
9,IS,IS,clean,"[b'AS', None]","[b'AS', None]"
10,IT,IT,clean,"[b'AT', None]","[b'AT', None]"
11,WAS,WAS,clean,"[b'AS', b'FS']","[b'AS', b'FS']"


## On noise 0 dBSNR

In [30]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_0SNR']  # Name of data splits to be used as validation set

low_dataset = Dataset(path, dev_split, None, 1)

updated_dataset = low_dataset.dataset.map(load_audio)

Using custom data configuration default-c9dd0aeaa110b3ae
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-1aa1c950d7d80a14.arrow


In [33]:
low_results = run_inference(model, updated_dataset)

  0%|          | 0/200 [00:00<?, ?it/s]

/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/src/module.py:197: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  feat_len = feat_len//4


In [34]:

low_results = pd.DataFrame(low_results)
low_results['condition'] = '0dB SNR'
top_match_guesses = low_results[(low_results['hyp'] == low_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

27 matched guesses; 13.5% correct


,hyp,truth,condition
5,THAT,THAT,0dB SNR
12,FOR,FOR,0dB SNR
14,THIS,THIS,0dB SNR
21,AS,AS,0dB SNR
28,WHAT,WHAT,0dB SNR


In [35]:

low_results['hyp_dmeta'] = low_results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
low_results['truth_dmeta'] = low_results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = low_results[(low_results['hyp_dmeta'] == low_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

33 matched guesses; 16.5% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
5,THAT,THAT,0dB SNR,"[b'0T', b'TT']","[b'0T', b'TT']"
12,FOR,FOR,0dB SNR,"[b'FR', None]","[b'FR', None]"
14,THIS,THIS,0dB SNR,"[b'0S', b'TS']","[b'0S', b'TS']"
18,ONE,ON,0dB SNR,"[b'AN', None]","[b'AN', None]"
21,AS,AS,0dB SNR,"[b'AS', None]","[b'AS', None]"


## On noise -5 dBSNR

In [36]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_-5SNR']  # Name of data splits to be used as validation set

high_dataset = Dataset(path, dev_split, None, 1)

updated_dataset = high_dataset.dataset.map(load_audio)

Using custom data configuration default-5683877c4c4495f8
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c3c5b179e5247f84.arrow


In [37]:
high_results = run_inference(model, updated_dataset)

  0%|          | 0/200 [00:00<?, ?it/s]

/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/src/module.py:197: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  feat_len = feat_len//4


In [38]:

high_results = pd.DataFrame(high_results)
high_results['condition'] = '-5dB'
top_match_guesses = high_results[(high_results['hyp'] == high_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(high_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

14 matched guesses; 7.0% correct


,hyp,truth,condition
28,WHAT,WHAT,-5dB
40,THERE,THERE,-5dB
48,WHEN,WHEN,-5dB
64,NOW,NOW,-5dB
66,HOW,HOW,-5dB


In [39]:

high_results['hyp_dmeta'] = high_results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
high_results['truth_dmeta'] = high_results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = high_results[(high_results['hyp_dmeta'] == high_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(high_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

17 matched guesses; 8.5% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
28,WHAT,WHAT,-5dB,"[b'AT', None]","[b'AT', None]"
40,THERE,THERE,-5dB,"[b'0R', b'TR']","[b'0R', b'TR']"
45,OH OH,AH,-5dB,"[b'A', None]","[b'A', None]"
48,WHEN,WHEN,-5dB,"[b'AN', None]","[b'AN', None]"
64,NOW,NOW,-5dB,"[b'N', b'NF']","[b'N', b'NF']"


In [ ]:
all_results = pd.concat([clean_results, low_results, high_results])

In [ ]:
all_results